In [0]:
# --------------------------------------------------------------
# Imports
# --------------------------------------------------------------
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import types as T
import random

# --------------------------------------------------------------
# Spark session (in Databricks a SparkSession already exists)
# --------------------------------------------------------------
spark = SparkSession.builder.getOrCreate()

In [0]:
# --------------------------------------------------------------
# 1. Employees & Departments
# --------------------------------------------------------------
NUM_EMPLOYEES = 150
DEPARTMENTS = ["Finance", "HR", "Engineering", "Marketing", "Operations"]
ACTIVITIES = ["logon", "file_open", "usb_connect", "email_external"]

# Create a DataFrame with employee_id, department
emp_df = (
    spark.range(1, NUM_EMPLOYEES + 1)                     # employee_id = 1 … 150
         .withColumnRenamed("id", "employee_id")
         .withColumn("department", 
                     F.element_at(
                         F.array(*[F.lit(d) for d in DEPARTMENTS]),
                         ((F.col("employee_id") % len(DEPARTMENTS)) + 1).cast("int")))
)

# Show a few rows (optional)
display(emp_df.limit(5))

employee_id,department
1,HR
2,Engineering
3,Marketing
4,Operations
5,Finance


In [0]:
# --------------------------------------------------------------
# 2. Time window (last 30 days, hourly granularity)
# --------------------------------------------------------------
START_DATE = "2023-10-01"   # adjust as you like
END_DATE   = "2023-10-30"

from datetime import date

# Create a DataFrame with one row per hour in the window
hours_df = (
    spark.range(0, ((date.fromisoformat(END_DATE) - date.fromisoformat(START_DATE)).days + 1) * 24)
         .withColumn("timestamp", 
                     F.expr(f"timestamp('{START_DATE}')") + F.make_interval(hours=F.col("id")))
         .select("timestamp")
)

# --------------------------------------------------------------
# 3. Cross join employees × hours → a row for every employee‑hour
# --------------------------------------------------------------
base_df = emp_df.crossJoin(hours_df)

# --------------------------------------------------------------
# 4. Randomly assign an activity type per row (≈ 1‑3 events per hour)
# --------------------------------------------------------------
# First create a small “event count” per employee‑hour (0‑2 extra events)
base_df = (
    base_df
    .withColumn("n_extra", F.floor(F.rand() * 2).cast("int"))  # 0‑1 extra events
)

# explode to have one row per event (the base event + possible extra)
events_df = (
    base_df
    .withColumn("event_seq", F.sequence(F.lit(0), F.col("n_extra")))
    .withColumn("event_seq", F.explode("event_seq"))
    .drop("n_extra", "event_seq")
)

# Random activity type per event
events_df = events_df.withColumn(
    "activity", 
    F.expr(f"element_at(array({', '.join(f'{a!r}' for a in ACTIVITIES)}), cast(floor(rand()*{len(ACTIVITIES)})+1 as int))")
)

# --------------------------------------------------------------
# 5. Add auxiliary columns (file size, email direction, etc.)
# --------------------------------------------------------------
events_df = (
    events_df
    # file size (bytes) – realistic only for file_open
    .withColumn("file_size_bytes",
        F.when(F.col("activity") == "file_open",
               F.floor(F.rand() * 5_000_000) + 10_000)   # 10KB‑5MB
         .otherwise(None)
    )
    # email direction – external vs internal (only for email_external)
    .withColumn("email_external",
        F.when(F.col("activity") == "email_external",
               F.when(F.rand() < 0.7, "external").otherwise("internal"))
         .otherwise(None)
    )
    # USB device id (only for usb_connect)
    .withColumn("usb_device_id",
        F.when(F.col("activity") == "usb_connect",
               F.concat(F.lit("USB-"), F.lpad(F.floor(F.rand()*9999).cast("string"), 4, "0")))
         .otherwise(None)
    )
)

# --------------------------------------------------------------
# 6. Confidential flag – baseline ~10 % true
# --------------------------------------------------------------
events_df = events_df.withColumn(
    "is_confidential",
    F.when(F.rand() < 0.10, True).otherwise(False)
)

# --------------------------------------------------------------
# 7. Keep only relevant columns for the bronze layer
# --------------------------------------------------------------
baseline_events = events_df.select(
    "employee_id",
    "department",
    "timestamp",
    "activity",
    "file_size_bytes",
    "email_external",
    "usb_device_id",
    "is_confidential"
)

display(baseline_events.limit(5))

employee_id,department,timestamp,activity,file_size_bytes,email_external,usb_device_id,is_confidential
1,HR,2023-10-01T00:00:00.000Z,email_external,null,external,null,false
1,HR,2023-10-01T00:00:00.000Z,logon,null,null,null,false
2,Engineering,2023-10-01T01:00:00.000Z,usb_connect,null,null,USB-1183,true
2,Engineering,2023-10-01T01:00:00.000Z,logon,null,null,null,false
3,Marketing,2023-10-01T02:00:00.000Z,logon,null,null,null,false


In [0]:
# --------------------------------------------------------------
# 2. Time window (last 30 days, hourly granularity)
# --------------------------------------------------------------
START_DATE = "2023-10-01"   # adjust as you like
END_DATE   = "2023-10-30"

from datetime import date

# Create a DataFrame with one row per hour in the window
hours_df = (
    spark.range(0, ((date.fromisoformat(END_DATE) - date.fromisoformat(START_DATE)).days + 1) * 24)
         .withColumn("timestamp", 
                     F.expr(f"timestamp('{START_DATE}')") + F.make_interval(hours=F.col("id")))
         .select("timestamp")
)

# --------------------------------------------------------------
# 3. Cross join employees × hours → a row for every employee‑hour
# --------------------------------------------------------------
base_df = emp_df.crossJoin(hours_df)

# --------------------------------------------------------------
# 4. Randomly assign an activity type per row (≈ 1‑3 events per hour)
# --------------------------------------------------------------
# First create a small “event count” per employee‑hour (0‑2 extra events)
base_df = (
    base_df
    .withColumn("n_extra", F.floor(F.rand() * 2).cast("int"))  # 0‑1 extra events
)

# explode to have one row per event (the base event + possible extra)
events_df = (
    base_df
    .withColumn("event_seq", F.sequence(F.lit(0), F.col("n_extra")))
    .withColumn("event_seq", F.explode("event_seq"))
    .drop("n_extra", "event_seq")
)

# Random activity type per event
events_df = events_df.withColumn(
    "activity", 
    F.expr(f"element_at(array({', '.join(f'{a!r}' for a in ACTIVITIES)}), cast(floor(rand()*{len(ACTIVITIES)})+1 as int))")
)

# --------------------------------------------------------------
# 5. Add auxiliary columns (file size, email direction, etc.)
# --------------------------------------------------------------
events_df = (
    events_df
    # file size (bytes) – realistic only for file_open
    .withColumn("file_size_bytes",
        F.when(F.col("activity") == "file_open",
               F.floor(F.rand() * 5_000_000) + 10_000)   # 10KB‑5MB
         .otherwise(None)
    )
    # email direction – external vs internal (only for email_external)
    .withColumn("email_external",
        F.when(F.col("activity") == "email_external",
               F.when(F.rand() < 0.7, "external").otherwise("internal"))
         .otherwise(None)
    )
    # USB device id (only for usb_connect)
    .withColumn("usb_device_id",
        F.when(F.col("activity") == "usb_connect",
               F.concat(F.lit("USB-"), F.lpad(F.floor(F.rand()*9999).cast("string"), 4, "0")))
         .otherwise(None)
    )
)

# --------------------------------------------------------------
# 6. Confidential flag – baseline ~10 % true
# --------------------------------------------------------------
events_df = events_df.withColumn(
    "is_confidential",
    F.when(F.rand() < 0.10, True).otherwise(False)
)

# --------------------------------------------------------------
# 7. Keep only relevant columns for the bronze layer
# --------------------------------------------------------------
baseline_events = events_df.select(
    "employee_id",
    "department",
    "timestamp",
    "activity",
    "file_size_bytes",
    "email_external",
    "usb_device_id",
    "is_confidential"
)

display(baseline_events.limit(5))

employee_id,department,timestamp,activity,file_size_bytes,email_external,usb_device_id,is_confidential
1,HR,2023-10-01T00:00:00.000Z,file_open,2894922,null,null,false
2,Engineering,2023-10-01T01:00:00.000Z,logon,null,null,null,false
2,Engineering,2023-10-01T01:00:00.000Z,email_external,null,external,null,false
3,Marketing,2023-10-01T02:00:00.000Z,logon,null,null,null,false
4,Operations,2023-10-01T03:00:00.000Z,usb_connect,null,null,USB-0803,false


In [0]:
# --------------------------------------------------------------
# 3. Choose three distinct insider employee IDs
# --------------------------------------------------------------
insider_ids = random.sample(range(1, NUM_EMPLOYEES + 1), 3)
print("Insider employee IDs:", insider_ids)

# Broadcast as a Spark DataFrame for later joins
insider_df = spark.createDataFrame([(i,) for i in insider_ids], ["employee_id"])

Insider employee IDs: [40, 24, 124]


In [0]:
from pyspark.sql import DataFrame
from datetime import date


def generate_insider_events(insider_id: int,
                            dept: str,
                            start_ts: str,
                            end_ts: str,
                            bulk_download_hours: int = 4,
                            after_hours_start: int = 20,
                            after_hours_end: int = 6,
                            cross_depts: list = None) -> DataFrame:
    """
    Returns a DataFrame with three hallmark behaviours for a given insider:
      • Bulk file_open events (many per hour)
      • USB connect events outside normal business hours (20‑06)
      • File_open events that belong to a *different* department.
    """
    # 1️⃣ Bulk downloads – pick a few random days and generate many file_open rows
    bulk_days = (
        spark.range(0, (date.fromisoformat(end_ts) - date.fromisoformat(start_ts)).days + 1)
             .withColumn("date", F.expr(f"date('{start_ts}')") + F.make_interval(days=F.col("id")))
    )
    bulk_days = bulk_days.sample(False, 0.2, seed=insider_id)   # ~20 % of days get bulk activity
    
    bulk_events = (
        bulk_days
        .withColumn("hour", F.floor(F.rand() * bulk_download_hours))   # concentrate in first N hours
        .withColumn("timestamp", F.expr("timestamp(date)") + F.make_interval(hours=F.col("hour")))
        .withColumn("activity", F.lit("file_open"))
        .withColumn("file_size_bytes",
                    F.floor(F.rand() * 500_000_000) + 50_000_000)      # 50 MB‑500 MB (bulk)
        .withColumn("is_confidential", F.lit(True))
        .withColumn("employee_id", F.lit(insider_id))
        .withColumn("department", F.lit(dept))
        .withColumn("email_external", F.lit(None).cast("string"))
        .withColumn("usb_device_id", F.lit(None).cast("string"))
        .select("employee_id","department","timestamp","activity",
                "file_size_bytes","email_external","usb_device_id","is_confidential")
    )
    
    # 2️⃣ After‑hours USB connects (random times between 20:00‑06:00)
    usb_events = (
        spark.range(0, 30)   # one event per day on average
        .withColumn("rand_day", F.floor(F.rand()*30))
        .withColumn("date", F.expr(f"date('{start_ts}')") + F.make_interval(days=F.col("rand_day")))
        .withColumn("hour",
            F.when(F.rand() < 0.5,
                   F.floor(F.rand()* (24 - after_hours_start)) + after_hours_start)   # 20‑23
            .otherwise(F.floor(F.rand()*after_hours_end))                               # 0‑5
        )
        .withColumn("minute", F.floor(F.rand()*60))
        .withColumn("timestamp",
            F.expr("timestamp(date) + make_interval(0,0,0,0,hour,minute,0)"))
        .withColumn("activity", F.lit("usb_connect"))
        .withColumn("usb_device_id",
                    F.concat(F.lit("USB-INSIDER-"), F.lpad(F.floor(F.rand()*9999).cast("string"),4,"0")))
        .withColumn("is_confidential", F.lit(True))
        .withColumn("employee_id", F.lit(insider_id))
        .withColumn("department", F.lit(dept))
        .withColumn("file_size_bytes", F.lit(None).cast("long"))
        .withColumn("email_external", F.lit(None).cast("string"))
        .select("employee_id","department","timestamp","activity",
                "file_size_bytes","email_external","usb_device_id","is_confidential")
    )
    
    # 3️⃣ Cross‑department file opens
    # Choose 2‑3 other departments to “snoop”
    other_depts = [d for d in DEPARTMENTS if d != dept]
    chosen_depts = random.sample(other_depts, k=random.randint(2,3))
    
    cross_events = (
        spark.range(0, len(chosen_depts) * 20)   # about 20 events per borrowed dept
        .withColumn("dept_idx", (F.col("id") % len(chosen_depts)).cast("int"))
        .withColumn("target_dept", F.element_at(F.array(*[F.lit(d) for d in chosen_depts]), F.col("dept_idx")+1))
        .withColumn("date", F.expr(f"date('{start_ts}')") + F.make_interval(days=F.floor(F.rand()*30)))
        .withColumn("hour", F.floor(F.rand()*8) + 9)   # normal office hours
        .withColumn("timestamp", F.expr("timestamp(date)") + F.make_interval(hours=F.col("hour")))
        .withColumn("activity", F.lit("file_open"))
        .withColumn("file_size_bytes", F.floor(F.rand()*2_000_000)+20_000)   # 20KB‑2MB
        .withColumn("is_confidential", F.lit(True))
        .withColumn("employee_id", F.lit(insider_id))
        .withColumn("department", F.col("target_dept"))    # the file belongs to a different dept
        .withColumn("email_external", F.lit(None).cast("string"))
        .withColumn("usb_device_id", F.lit(None).cast("string"))
        .select("employee_id","department","timestamp","activity",
                "file_size_bytes","email_external","usb_device_id","is_confidential")
    )
    
    return bulk_events.unionByName(usb_events).unionByName(cross_events)

In [0]:
# --------------------------------------------------------------
# 4. Build insider DataFrames
# --------------------------------------------------------------
insider_events = None

for i, ins_id in enumerate(insider_ids):
    # Lookup the employee's home department
    emp_dept = emp_df.filter(F.col("employee_id") == ins_id).select("department").first()["department"]
    
    # Generate events for this insider (30‑day window)
    ie = generate_insider_events(
            insider_id = ins_id,
            dept = emp_dept,
            start_ts = START_DATE,
            end_ts   = END_DATE
        )
    insider_events = ie if insider_events is None else insider_events.unionByName(ie)

# --------------------------------------------------------------
# 5. Union insider events with the baseline (deduplicate if needed)
# --------------------------------------------------------------
full_events = baseline_events.unionByName(insider_events)

# Optional sanity check
display(full_events.groupBy("activity").count().orderBy("activity"))

activity,count
email_external,40394
file_open,40533
logon,40484
usb_connect,40926


In [0]:
# --------------------------------------------------------------
# 6. Persist as Delta – Bronze layer
# --------------------------------------------------------------
# DBFS mounts (/mnt/...) are disabled in this workspace — persist as a
# Unity Catalog managed Delta table instead
full_events.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.default.bronze_events")

print("✅ Delta table 'workspace.bronze.bronze_events' successfully created.")

✅ Delta table 'workspace.bronze.bronze_events' successfully created.


In [0]:
full_events.write.mode("overwrite").saveAsTable("workspace.default.bronze_events")

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-5273094988870636>, line 1
----> 1 full_events.write.mode("overwrite").saveAsTable("workspace.default.bronze_events")

File <command-8912478571385663>, line 22
     17     insider_events = ie if insider_events is None else insider_events.unionByName(ie)
     19 # --------------------------------------------------------------
     20 # 5. Union insider events with the baseline (deduplicate if needed)
     21 # --------------------------------------------------------------
---> 22 full_events = baseline_events.unionByName(insider_events)
     24 # Optional sanity check
     25 display(full_events.groupBy("activity").count().orderBy("activity"))

File <command-8912478571385660>, line 81
     73 events_df = events_df.withColumn(
     74     "is_confidential",
     75     F.when(F.rand() < 0.10, True).otherwise(False)
     76 )
